In [1]:
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"/content/spam.csv", encoding='latin-1')

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
df = df[['v1','v2']]
df.columns = ['label','message']
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df['label'] = df['label'].map({
    'ham':0,
    'spam':1
})

In [5]:
df['message'] = df['message'].str.lower()

In [6]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['message'] = df['message'].apply(remove_punctuation)

In [7]:
nltk.download('stopwords')
stop_words = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [8]:
df['message'] = df['message'].apply(
    lambda x: " ".join(
        word for word in x.split()
        if word not in stop_words
    )
)

In [9]:
stemmer = PorterStemmer()

df['message'] = df['message'].apply(
    lambda x: " ".join(
        stemmer.stem(word)
        for word in x.split()
    )
)

In [10]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['message'])
y = df['label']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [13]:
y_pred = model.predict(X_test)

In [14]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9614349775784753


In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.71      0.83       150

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [16]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[965   0]
 [ 43 107]]


In [17]:
sample = ["Congratulations! You have won a free iPhone. Click here now."]

sample_vector = vectorizer.transform(sample)
prediction = model.predict(sample_vector)

if prediction[0] == 1:
    print("Spam Email")
else:
    print("Not Spam")

Not Spam


In [18]:
print("\n========== Spam Analysis Report ==========")

print("Total Emails :", len(df))
print("Spam Emails  :", sum(df['label']==1))
print("Ham Emails   :", sum(df['label']==0))

print("\nSpam Percentage : {:.2f}%".format(
    (sum(df['label']==1)/len(df))*100))

print("Ham Percentage  : {:.2f}%".format(
    (sum(df['label']==0)/len(df))*100))

print("\nModel Accuracy : {:.2f}%".format(accuracy*100))


========== Spam Analysis Report ==========
Total Emails : 5572
Spam Emails  : 747
Ham Emails   : 4825

Spam Percentage : 13.41%
Ham Percentage  : 86.59%

Model Accuracy : 96.14%
